# 04 - Results and Trade-Off Analysis
Create cost/risk trade-off points and visualize the Pareto front.

## Step 1: Build analysis set
Load data, predict delays, define demand, and simulate scenarios.

In [1]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd

# Ensure project root (folder containing src/) is importable in notebook sessions.
project_root = Path.cwd().resolve()
while not (project_root / "src").exists() and project_root != project_root.parent:
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.data_generation import default_component_demand
from src.delay_predictor import train_delay_model, predict_delays
from src.graph_visualization import plot_pareto_front
from src.optimizer import generate_delay_scenarios, optimize_procurement

data_path = project_root / "data" / "supply_chain_dataset.csv"
df = pd.read_csv(data_path)
artifacts = train_delay_model(df)
df['predicted_delay_days'] = predict_delays(artifacts.pipeline, df)
demand = default_component_demand(df)
scenarios = generate_delay_scenarios(df['predicted_delay_days'].to_numpy(), artifacts.metrics['residual_std'], 80, 42)

## Step 2: Generate Pareto points
Sweep risk weight while keeping a fixed carbon weight, then collect cost-risk-carbon outputs.

In [2]:
rows = []
carbon_weight = 0.1
for w in np.linspace(0, 1, 11):
    _, s = optimize_procurement(
        df=df,
        demand=demand,
        budget=2_000_000,
        deadline_days=28,
        delay_penalty=4.0,
        delay_scenarios=scenarios,
        risk_weight=float(w),
        carbon_weight=carbon_weight,
        scenario_confidence=0.9,
        delay_variability_cap=3.0,
        solver_time_limit=10,
    )
    if s['status'] == 'Optimal':
        rows.append({
            'risk_weight': float(w),
            'carbon_weight': float(carbon_weight),
            'total_cost': s['total_cost'],
            'risk_score': s['risk_score'],
            'carbon_score_kg': s['carbon_score_kg'],
        })
pareto_df = pd.DataFrame(rows)
pareto_df

,risk_weight,carbon_weight,total_cost,risk_score,carbon_score_kg
0,0.0,0.1,2.046761e+06,7.559814,12927.612649
1,0.1,0.1,2.046761e+06,7.559814,12927.612649
2,0.2,0.1,2.046761e+06,7.559814,12927.612649
3,0.3,0.1,2.046761e+06,7.559814,12927.612649
4,0.4,0.1,2.046761e+06,7.559814,12927.612649
5,0.5,0.1,2.046761e+06,7.559814,12927.612649
6,0.6,0.1,2.047914e+06,7.597175,12926.926807
7,0.7,0.1,2.047914e+06,7.597175,12926.926807
8,0.8,0.1,2.047914e+06,7.597175,12926.926807
9,0.9,0.1,2.047914e+06,7.597175,12926.926807


## Step 3: Export chart
Save the interactive Pareto front HTML.

In [4]:
plot_pareto_front(pareto_df, '../results/pareto_front.html')
print('Saved ../results/pareto_front.html')

Saved ../results/pareto_front.html
